In [ ]:
from dataset.utils import TextFRCT
from dataset.demonstrations import VANILLA_DEMONSTRATIONS
from dataset.questions import VANILLA_QUESTIONS
from IPython.display import clear_output
from utils import build_model, ask_llm
import time
import csv
import os

RETRY_NUM = 5
MODEL = 'gpt-4o-mini-2024-07-18'
FILENAME = os.path.join("data", f"{MODEL}.csv")
fieldnames = ["input", "output"]
os.makedirs('data', exist_ok=True)

dataset = TextFRCT()
CLIENT = build_model(MODEL)

In [ ]:
with open(FILENAME, mode="w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

for idx, prompt in enumerate(dataset.build_prompt(VANILLA_QUESTIONS, VANILLA_DEMONSTRATIONS)):
    clear_output(wait=True)
    print(f"{idx}\n\n{prompt}")
    
    for attempt in range(RETRY_NUM):
        try:
            response_text = ask_llm(CLIENT, MODEL, [{"role": "user", "content": prompt}])
            if response_text:
                print(f"\n\n>>>>>>>>\n{response_text}")
                break
        except Exception as e:
            reason = f"Failed to parse JSON: {e}"
            print(f"{reason}; Attempt {attempt+1} failed; retrying...")
            time.sleep(1)
    
    with open(FILENAME, mode="a", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writerow({"input": prompt, "output": response_text})

In [ ]:
predictions = []
with open(FILENAME, mode="r", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        predictions.append(row["output"])

results = dataset.evaluate(predictions, FILENAME.replace('.csv', '_processed.csv'))